In [ ]:
# ============================================================
# PROJETO BIG DATA - SEGURANÇA PÚBLICA RJ
# Etapa 1: Pipeline inicial - Base CISP (Roubo/Furto) + banco SQLite
# Etapa 2: Análise descritiva, detecção de outliers e regressão linear
# Etapa 3: Apresentação de conclusões (PowerPoint)
# Município do Rio de Janeiro | 2005-2015
# ============================================================

import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from google.colab import files

# ------------------------------------------------------------
# 1. Carregar a base de dados
# ------------------------------------------------------------
url = "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv"
df = pd.read_csv(url, sep=";", encoding="latin1", low_memory=False)

print("Shape original:", df.shape)
print("Anos disponíveis:", sorted(map(int, df["ano"].unique())))
print("Municípios únicos:", df["munic"].nunique())

# ------------------------------------------------------------
# 2. Definir filtros do projeto
# ------------------------------------------------------------
MUNICIPIO = "Rio de Janeiro"
ANOS = list(range(2005, 2016))   # 2005 até 2015

# ------------------------------------------------------------
# 3. Aplicar filtros
# ------------------------------------------------------------
df_rj = df[
    (df["munic"] == MUNICIPIO) &
    (df["ano"].isin(ANOS))
].copy()

print("\nShape após filtro (Rio + 2005-2015):", df_rj.shape)
print("CISP únicos:", df_rj["cisp"].nunique())

# ------------------------------------------------------------
# 4. Selecionar colunas relevantes
# ------------------------------------------------------------
COLS_ID = [
    "cisp", "mes", "ano", "mes_ano",
    "aisp", "risp", "munic", "mcirc", "regiao"
]

COLS_ROUBO = [
    "roubo_transeunte", "roubo_celular", "roubo_em_coletivo",
    "roubo_rua", "roubo_veiculo", "roubo_carga", "roubo_comercio",
    "roubo_residencia", "roubo_banco", "roubo_cx_eletronico",
    "roubo_conducao_saque", "roubo_apos_saque", "roubo_bicicleta",
    "outros_roubos", "total_roubos"
]

COLS_FURTO = [
    "furto_veiculos", "furto_transeunte", "furto_coletivo",
    "furto_celular", "furto_bicicleta", "outros_furtos", "total_furtos"
]

COLS_KEEP = COLS_ID + COLS_ROUBO + COLS_FURTO
cols_existentes = [c for c in COLS_KEEP if c in df_rj.columns]
df_rj = df_rj[cols_existentes].copy()

print("Colunas mantidas:", len(df_rj.columns))

# ------------------------------------------------------------
# 5. Limpeza básica
# ------------------------------------------------------------
# roubo_bicicleta e furto_bicicleta só passaram a ser registrados em 2014.
# Antes disso o valor é vazio ("não coletado"), não zero. Usamos o tipo
# inteiro anulável "Int64" para manter esses vazios como <NA>; assim as
# médias, medianas e desvios não são puxados para baixo por zeros falsos.
cols_numericas = [c for c in (COLS_ROUBO + COLS_FURTO) if c in df_rj.columns]

for col in cols_numericas:
    df_rj[col] = pd.to_numeric(df_rj[col], errors="coerce").astype("Int64")

df_rj["cisp"] = df_rj["cisp"].astype("int64")
df_rj["mes"]  = df_rj["mes"].astype("int64")
df_rj["ano"]  = df_rj["ano"].astype("int64")

print("\nValores não coletados (NA) por coluna:")
print(df_rj[cols_numericas].isna().sum()[lambda s: s > 0])

# ------------------------------------------------------------
# 6. Verificação rápida
# ------------------------------------------------------------
print("\n=== Validação ===")
print("Anos presentes:", sorted(map(int, df_rj["ano"].unique())))
print("Total de registros (CISP x mês):", len(df_rj))
print("Soma total_roubos (2005-2015):", df_rj["total_roubos"].sum())
print("Soma total_furtos (2005-2015):", df_rj["total_furtos"].sum())

assert not df_rj.duplicated(["cisp", "ano", "mes"]).any(), "Há CISP x mês duplicado"

print("\nAmostra:")
display(df_rj[["cisp", "ano", "mes", "total_roubos", "total_furtos", "roubo_rua", "furto_veiculos"]].head(10))

# ------------------------------------------------------------
# 7. Carregar a base processada no banco de dados (SQLite)
# ------------------------------------------------------------
# SQLite já vem com o Python: o banco inteiro é um único arquivo .db.
# Três tabelas:
#   delegacias  -> uma linha por CISP (dados que não mudam no período)
#   ocorrencias -> uma linha por CISP x mês, com as contagens de crimes
#   populacao   -> população do município por ano (IBGE), para as taxas
# A AISP fica em ocorrencias, e não em delegacias, porque várias CISPs
# mudaram de AISP na reorganização de 2011.
#
# Atenção no Colab: o arquivo some quando a sessão reinicia. Baixe o .db
# no final (item 12.1) ou salve no Google Drive.
ARQUIVO_DB = "seguranca_rj.db"

# População residente do município do Rio de Janeiro (IBGE/SIDRA):
# estimativas anuais (tabela 6579), Contagem 2007 (tabela 793) e Censo 2010 (tabela 202).
POPULACAO_RIO = {
    2005: 6_094_183, 2006: 6_136_652, 2007: 6_093_472, 2008: 6_161_047,
    2009: 6_186_710, 2010: 6_320_446, 2011: 6_355_949, 2012: 6_390_290,
    2013: 6_429_923, 2014: 6_453_682, 2015: 6_476_631,
}

COLS_DELEGACIA = ["cisp", "risp", "regiao", "munic", "mcirc"]
COLS_OCORRENCIA = ["cisp", "ano", "mes", "mes_ano", "aisp"] + cols_numericas

assert df_rj.groupby("cisp")[COLS_DELEGACIA[1:]].nunique().max().max() == 1, \
    "Algum dado de delegacia muda no período; ele deveria ir para ocorrencias"

if os.path.exists(ARQUIVO_DB):          # recria o banco do zero a cada execução
    os.remove(ARQUIVO_DB)
conn = sqlite3.connect(ARQUIVO_DB)
conn.execute("PRAGMA foreign_keys = ON")

colunas_crimes = ",\n    ".join(f"{c:<20} INTEGER" for c in cols_numericas)
conn.executescript(f"""
CREATE TABLE delegacias (
    cisp    INTEGER PRIMARY KEY,
    risp    INTEGER,
    regiao  TEXT,
    munic   TEXT,
    mcirc   INTEGER
);

CREATE TABLE ocorrencias (
    cisp    INTEGER NOT NULL REFERENCES delegacias (cisp),
    ano     INTEGER NOT NULL,
    mes     INTEGER NOT NULL CHECK (mes BETWEEN 1 AND 12),
    mes_ano TEXT,
    aisp    INTEGER,
    {colunas_crimes},
    PRIMARY KEY (cisp, ano, mes)
);

CREATE TABLE populacao (
    ano       INTEGER PRIMARY KEY,
    populacao INTEGER NOT NULL
);
""")

# Valores <NA> (bicicleta antes de 2014) viram NULL no banco
df_rj[COLS_DELEGACIA].drop_duplicates("cisp").to_sql("delegacias", conn, if_exists="append", index=False)
df_rj[COLS_OCORRENCIA].to_sql("ocorrencias", conn, if_exists="append", index=False)
pd.DataFrame(list(POPULACAO_RIO.items()), columns=["ano", "populacao"]) \
    .to_sql("populacao", conn, if_exists="append", index=False)
conn.commit()

# Conferência: o banco precisa ter exatamente os mesmos dados do DataFrame
conferencia = pd.read_sql(
    "SELECT COUNT(*) AS registros, SUM(total_roubos) AS roubos, SUM(total_furtos) AS furtos "
    "FROM ocorrencias", conn
).iloc[0]
assert conferencia["registros"] == len(df_rj)
assert conferencia["roubos"] == df_rj["total_roubos"].sum()
assert conferencia["furtos"] == df_rj["total_furtos"].sum()

print(f"\nBanco criado: {ARQUIVO_DB}")
for tabela in ["delegacias", "ocorrencias", "populacao"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"  {tabela:<12} {n:>6} linhas")

# ------------------------------------------------------------
# 7.1 Consultas SQL de exemplo
# ------------------------------------------------------------
CONSULTAS = {
    "Roubos e furtos por ano": """
        SELECT ano,
               SUM(total_roubos) AS roubos,
               SUM(total_furtos) AS furtos
        FROM ocorrencias
        GROUP BY ano
        ORDER BY ano
    """,
    "5 delegacias com mais furtos no período": """
        SELECT o.cisp, d.risp,
               SUM(o.total_furtos)        AS furtos,
               ROUND(AVG(o.total_furtos)) AS media_mensal
        FROM ocorrencias o
        JOIN delegacias d ON d.cisp = o.cisp
        GROUP BY o.cisp, d.risp
        ORDER BY furtos DESC
        LIMIT 5
    """,
    "Média de furtos da cidade por mês do ano": """
        SELECT mes, ROUND(AVG(furtos_cidade)) AS media_furtos
        FROM (SELECT ano, mes, SUM(total_furtos) AS furtos_cidade
              FROM ocorrencias
              GROUP BY ano, mes)
        GROUP BY mes
        ORDER BY mes
    """,
}

for titulo_consulta, sql in CONSULTAS.items():
    print(f"\n=== SQL: {titulo_consulta} ===")
    display(pd.read_sql(sql, conn))


# ============================================================
# ETAPA 2 - ANÁLISE DESCRITIVA, OUTLIERS E REGRESSÃO LINEAR
# ============================================================
# Duas unidades de análise:
#   (a) CISP x mês  -> cada linha da base (uma delegacia em um mês)
#   (b) Município x mês -> soma de todas as CISPs em cada mês (132 meses)
# O desvio padrão é o amostral (ddof=1), padrão do pandas.

COLS_PRINCIPAIS = ["total_roubos", "total_furtos"]

# Trabalhamos em float para os cálculos estatísticos (NA vira NaN e é ignorado)
dados = df_rj[cols_numericas].astype("float64")


def resumo_estatistico(frame):
    """Média, mediana, desvio padrão e medidas de posição de cada coluna."""
    q1 = frame.quantile(0.25)
    q3 = frame.quantile(0.75)
    media = frame.mean()
    desvio = frame.std()
    return pd.DataFrame({
        "n":             frame.count(),
        "media":         media,
        "mediana":       frame.median(),
        "desvio_padrao": desvio,
        "cv_%":          desvio / media * 100,   # coeficiente de variação
        "minimo":        frame.min(),
        "q1":            q1,
        "q3":            q3,
        "maximo":        frame.max(),
        "iqr":           q3 - q1,
        "assimetria":    frame.skew(),
    }).round(2)


# ------------------------------------------------------------
# 8. Estatísticas descritivas - nível CISP x mês
# ------------------------------------------------------------
stats_cisp = resumo_estatistico(dados)

print("\n=== Estatísticas descritivas (CISP x mês) ===")
display(stats_cisp)

# ------------------------------------------------------------
# 9. Estatísticas descritivas - totais mensais do município
# ------------------------------------------------------------
# Os totais mensais vêm direto do banco. O SUM do SQL ignora NULL e devolve
# NULL quando o mês inteiro é NULL (ex.: bicicleta antes de 2014), em vez de 0.
somas_sql = ",\n       ".join(f"SUM({c}) AS {c}" for c in cols_numericas)
mensal = pd.read_sql(f"""
    SELECT ano, mes,
       {somas_sql}
    FROM ocorrencias
    GROUP BY ano, mes
    ORDER BY ano, mes
""", conn)
mensal[cols_numericas] = mensal[cols_numericas].astype("float64")
mensal["data"] = pd.to_datetime(
    mensal["ano"].astype(str) + "-" + mensal["mes"].astype(str) + "-01"
)

stats_mensal = resumo_estatistico(mensal[cols_numericas])

print("\n=== Estatísticas descritivas (total mensal do município) ===")
display(stats_mensal)

# Média, mediana e desvio padrão dos totais mensais, ano a ano
stats_por_ano = (
    mensal.groupby("ano")[COLS_PRINCIPAIS]
    .agg(["mean", "median", "std"])
    .round(1)
)
stats_por_ano.columns = [
    f"{col}_{ {'mean': 'media', 'median': 'mediana', 'std': 'desvio'}[f] }"
    for col, f in stats_por_ano.columns
]

print("\n=== Totais mensais do município por ano ===")
display(stats_por_ano)

# ------------------------------------------------------------
# 9.1 Taxa por 100 mil habitantes (município, por ano)
# ------------------------------------------------------------
# A cidade cresceu no período; a taxa mostra se o crime aumentou mais
# do que a população. Calculada no banco, juntando (JOIN) as ocorrências
# com a tabela de população do IBGE (item 7).
taxas = pd.read_sql("""
    SELECT o.ano,
           p.populacao,
           SUM(o.total_roubos) AS total_roubos,
           SUM(o.total_furtos) AS total_furtos,
           ROUND(SUM(o.total_roubos) * 100000.0 / p.populacao, 1) AS taxa_total_roubos_100mil,
           ROUND(SUM(o.total_furtos) * 100000.0 / p.populacao, 1) AS taxa_total_furtos_100mil
    FROM ocorrencias o
    JOIN populacao p ON p.ano = o.ano
    GROUP BY o.ano, p.populacao
    ORDER BY o.ano
""", conn, index_col="ano")

print("\n=== Taxa anual por 100 mil habitantes ===")
display(taxas)

# ------------------------------------------------------------
# 9.2 Composição: quais tipos de crime pesam mais no total?
# ------------------------------------------------------------
# roubo_rua já é a soma de roubo_transeunte, roubo_celular e roubo_em_coletivo.
# Por isso entra só ele aqui: somar os quatro contaria os mesmos casos duas vezes.
TIPOS_ROUBO = {
    "roubo_rua": "Na rua", "roubo_veiculo": "Veículo", "roubo_carga": "Carga",
    "roubo_comercio": "Comércio", "roubo_residencia": "Residência", "roubo_banco": "Banco",
    "roubo_cx_eletronico": "Caixa eletrônico", "roubo_conducao_saque": "Condução de saque",
    "roubo_apos_saque": "Após saque", "roubo_bicicleta": "Bicicleta", "outros_roubos": "Outros",
}
TIPOS_FURTO = {
    "furto_veiculos": "Veículo", "furto_transeunte": "Transeunte", "furto_coletivo": "Coletivo",
    "furto_celular": "Celular", "furto_bicicleta": "Bicicleta", "outros_furtos": "Outros",
}


def composicao(tipos, total):
    """Total de cada tipo no período e sua participação no total do crime."""
    soma = mensal[list(tipos)].sum()
    assert soma.sum() == mensal[total].sum(), f"Os tipos não somam {total}"
    tabela = pd.DataFrame({
        "tipo": soma.index.map(tipos),
        "ocorrencias": soma.values,
        "participacao_%": (soma / soma.sum() * 100).round(1).values,
    }, index=soma.index.rename("variavel"))
    return tabela.sort_values("ocorrencias", ascending=False)


composicao_roubos = composicao(TIPOS_ROUBO, "total_roubos")
composicao_furtos = composicao(TIPOS_FURTO, "total_furtos")

print("\n=== Composição dos roubos ===")
display(composicao_roubos)
print("\n=== Composição dos furtos ===")
display(composicao_furtos)

# ------------------------------------------------------------
# 9.3 Concentração: o crime está espalhado ou em poucas delegacias?
# ------------------------------------------------------------
# Atenção: 3 CISPs não têm os 132 meses (delegacias criadas ou divididas no
# período). A soma da cidade não muda, mas o total delas fica menor.
por_cisp = pd.read_sql("""
    SELECT cisp,
           COUNT(*)          AS meses,
           SUM(total_roubos) AS roubos,
           SUM(total_furtos) AS furtos
    FROM ocorrencias
    GROUP BY cisp
""", conn)
for col in ["roubos", "furtos"]:
    por_cisp[f"{col}_%"] = (por_cisp[col] / por_cisp[col].sum() * 100).round(1)
por_cisp = por_cisp.sort_values("furtos", ascending=False).reset_index(drop=True)


def parcela_maiores(coluna, n):
    """Parcela do total (em fração) que cabe às n delegacias com mais casos."""
    return por_cisp[coluna].nlargest(n).sum() / por_cisp[coluna].sum()


print("\n=== Delegacias com mais furtos ===")
display(por_cisp.head(10))
print(f"As 5 maiores concentram {parcela_maiores('furtos', 5):.1%} dos furtos "
      f"e {parcela_maiores('roubos', 5):.1%} dos roubos.")

# ------------------------------------------------------------
# 9.4 Correlação entre os totais mensais
# ------------------------------------------------------------
# Pearson mede relação linear; Spearman usa postos (ordens), então é menos
# afetado pelos picos de Carnaval. Correlação não indica causa.
COLS_CORRELACAO = ["total_roubos", "total_furtos", "roubo_rua", "roubo_veiculo",
                   "furto_transeunte", "furto_celular"]
correlacao = mensal[COLS_CORRELACAO].corr(method="pearson").round(2)
corr_spearman = mensal["total_roubos"].corr(mensal["total_furtos"], method="spearman")

print("\n=== Correlação de Pearson (totais mensais) ===")
display(correlacao)
print(f"Roubos x furtos: Pearson = {correlacao.loc['total_roubos', 'total_furtos']:.2f} | "
      f"Spearman = {corr_spearman:.2f}")

# ------------------------------------------------------------
# 10. Detecção de outliers
# ------------------------------------------------------------
# Dois métodos:
#   - IQR (Tukey): outlier se valor < Q1 - 1,5*IQR ou > Q3 + 1,5*IQR.
#     Robusto, não depende de normalidade -> método principal, porque
#     contagens de crimes são assimétricas.
#   - Z-score: outlier se |valor - média| / desvio > 3.
#     Supõe distribuição aproximadamente normal -> usado como comparação.
#
# No nível CISP x mês, o IQR global marca sobretudo as delegacias grandes
# (Centro, Copacabana...), que são "outliers" o tempo todo. Por isso
# também calculamos o IQR dentro de cada CISP: um mês atípico para
# aquela delegacia, comparado com o próprio histórico dela.

def limites_iqr(serie_ou_grupo, k=1.5):
    q1 = serie_ou_grupo.quantile(0.25)
    q3 = serie_ou_grupo.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def outlier_iqr(frame, k=1.5):
    inf, sup = limites_iqr(frame, k)
    return frame.lt(inf) | frame.gt(sup)


def outlier_zscore(frame, limite=3):
    z = (frame - frame.mean()) / frame.std()
    return z.abs().gt(limite)


def outlier_iqr_por_grupo(frame, grupos, k=1.5):
    g = frame.groupby(grupos)
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    return frame.lt(q1 - k * iqr) | frame.gt(q3 + k * iqr)


# 10.1 Resumo de outliers por variável (nível CISP x mês)
out_iqr_global = outlier_iqr(dados)
out_zscore     = outlier_zscore(dados)
out_iqr_cisp   = outlier_iqr_por_grupo(dados, df_rj["cisp"])

resumo_outliers = pd.DataFrame({
    "n_validos":         dados.count(),
    "iqr_global":        out_iqr_global.sum(),
    "iqr_global_%":      out_iqr_global.sum() / dados.count() * 100,
    "zscore_>3":         out_zscore.sum(),
    "zscore_%":          out_zscore.sum() / dados.count() * 100,
    "iqr_por_cisp":      out_iqr_cisp.sum(),
    "iqr_por_cisp_%":    out_iqr_cisp.sum() / dados.count() * 100,
}).round(2)

print("\n=== Quantidade de outliers por variável (CISP x mês) ===")
display(resumo_outliers)

# 10.2 Meses atípicos de cada delegacia (total de roubos e furtos)
detalhe_outliers = []
for col in COLS_PRINCIPAIS:
    marcados = df_rj.loc[out_iqr_cisp[col], ["cisp", "ano", "mes", col]].copy()
    marcados["mediana_da_cisp"] = (
        dados.groupby(df_rj["cisp"])[col].transform("median")[out_iqr_cisp[col]]
    )
    marcados["variavel"] = col
    marcados = marcados.rename(columns={col: "valor"})
    detalhe_outliers.append(marcados)

detalhe_outliers = (
    pd.concat(detalhe_outliers, ignore_index=True)
    .sort_values(["variavel", "cisp", "ano", "mes"])
)

print("\n=== Meses atípicos por CISP (IQR dentro da delegacia) ===")
print("Total de registros marcados:", len(detalhe_outliers))
display(detalhe_outliers.head(20))

print("\nCISPs com mais meses atípicos em total_roubos:")
display(
    detalhe_outliers.query("variavel == 'total_roubos'")["cisp"]
    .value_counts().head(10).rename("meses_atipicos")
)

# 10.3 Meses atípicos para o município como um todo
print("\n=== Meses atípicos do município (total mensal) ===")
SIM_NAO = {True: "sim", False: "não"}
outliers_municipio = []
for col in COLS_PRINCIPAIS:
    inf, sup = limites_iqr(mensal[col])
    flag_iqr = outlier_iqr(mensal[col])
    flag_z   = outlier_zscore(mensal[col])
    print(f"\n{col}: limites IQR = [{inf:,.0f} ; {sup:,.0f}] | "
          f"outliers IQR = {flag_iqr.sum()} | outliers z-score = {flag_z.sum()}")
    marcados = flag_iqr | flag_z
    tabela = mensal.loc[marcados, ["ano", "mes", col]].rename(columns={col: "valor"})
    tabela.insert(0, "variavel", col)
    tabela["limite_inferior_iqr"] = inf
    tabela["limite_superior_iqr"] = sup
    tabela["outlier_iqr"]    = flag_iqr[marcados].map(SIM_NAO)
    tabela["outlier_zscore"] = flag_z[marcados].map(SIM_NAO)
    if not tabela.empty:
        display(tabela)
    outliers_municipio.append(tabela)

outliers_municipio = pd.concat(outliers_municipio, ignore_index=True)

# ------------------------------------------------------------
# 11. Visualização - boxplots dos totais mensais por ano
# ------------------------------------------------------------
# Um gráfico por variável (mesma escala não faz sentido para roubos e furtos).
# Os pontos fora dos "bigodes" são os outliers pelo critério de 1,5*IQR,
# calculado dentro de cada ano (meses atípicos para aquele ano). Por isso
# eles podem diferir dos outliers da série inteira, calculados no item 10.3.
COR_SERIE   = "#2a78d6"
COR_TEXTO   = "#52514e"
COR_GRADE   = "#e5e4e0"
COR_FUNDO   = "#fcfcfb"

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True, facecolor=COR_FUNDO)

for ax, col, titulo in zip(
    axes, COLS_PRINCIPAIS,
    ["Total de roubos por mês - Município do Rio de Janeiro",
     "Total de furtos por mês - Município do Rio de Janeiro"],
):
    por_ano = [mensal.loc[mensal["ano"] == a, col].dropna() for a in ANOS]
    ax.boxplot(
        por_ano,
        tick_labels=[str(a) for a in ANOS],
        widths=0.5,
        patch_artist=True,
        boxprops=dict(facecolor=COR_SERIE, alpha=0.25, edgecolor=COR_SERIE, linewidth=1.5),
        medianprops=dict(color=COR_SERIE, linewidth=2),
        whiskerprops=dict(color=COR_TEXTO, linewidth=1),
        capprops=dict(color=COR_TEXTO, linewidth=1),
        flierprops=dict(marker="o", markersize=7, markerfacecolor=COR_SERIE,
                        markeredgecolor=COR_FUNDO, markeredgewidth=1.5),
    )
    ax.set_facecolor(COR_FUNDO)
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_ylabel("ocorrências no mês", color=COR_TEXTO)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", ".")))
    ax.grid(axis="y", color=COR_GRADE, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=COR_TEXTO)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(COR_GRADE)

axes[-1].set_xlabel("ano", color=COR_TEXTO)
fig.tight_layout()
fig.savefig("boxplot_roubos_furtos_por_ano.png", dpi=150)
plt.show()

# ------------------------------------------------------------
# 11.1 Distribuição dos totais mensais (histogramas)
# ------------------------------------------------------------
# Faixas iguais para roubos e furtos, para os dois gráficos serem comparáveis.
LARGURA_FAIXA = 500
faixas = np.arange(
    mensal[COLS_PRINCIPAIS].min().min() // LARGURA_FAIXA * LARGURA_FAIXA,
    mensal[COLS_PRINCIPAIS].max().max() + LARGURA_FAIXA,
    LARGURA_FAIXA,
)
distribuicao = pd.DataFrame({
    col: pd.cut(mensal[col], faixas, right=False).value_counts(sort=False).values
    for col in COLS_PRINCIPAIS
}, index=pd.Index(faixas[:-1].astype(int), name="inicio_da_faixa"))


def estilo_eixo(ax, titulo, rotulo_y):
    ax.set_facecolor(COR_FUNDO)
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_ylabel(rotulo_y, color=COR_TEXTO)
    ax.grid(axis="y", color=COR_GRADE, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=COR_TEXTO)
    for lado in ["top", "right", "left"]:
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(COR_GRADE)


COR_FURTO_G = "#eb6834"
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, facecolor=COR_FUNDO)
for ax, col, nome, cor in zip(axes, COLS_PRINCIPAIS, ["Roubos", "Furtos"], [COR_SERIE, COR_FURTO_G]):
    ax.hist(mensal[col], bins=faixas, color=cor, alpha=0.8, edgecolor=COR_FUNDO)
    ax.axvline(mensal[col].mean(), color="#0b0b0b", linewidth=1.5, label="média")
    ax.axvline(mensal[col].median(), color="#0b0b0b", linewidth=1.5, linestyle="--", label="mediana")
    estilo_eixo(ax, f"{nome}: total mensal do município", "quantidade de meses")
    ax.set_xlabel("ocorrências no mês", color=COR_TEXTO)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("histograma_roubos_furtos.png", dpi=150)
plt.show()

# ------------------------------------------------------------
# 11.2 Regressão linear - preparação das variáveis
# ------------------------------------------------------------
# A regressão linear estima uma equação do tipo y = b0 + b1*x1 + b2*x2 + erro,
# escolhendo os coeficientes que minimizam a soma dos erros ao quadrado (MQO).
# A unidade de análise é o total mensal do município (132 meses).
#
# Em séries temporais, o erro de um mês costuma lembrar o do mês anterior
# (autocorrelação). Isso não muda os coeficientes, mas deixa os erros padrão
# comuns otimistas demais. Por isso usamos erros padrão HAC (Newey-West),
# que corrigem esse problema, com defasagem de 12 meses.

# Mês em que caiu a terça-feira de Carnaval em cada ano.
# Em 2011 e 2014 foi em março: exatamente os anos em que o pico de furtos foi em março.
MES_CARNAVAL = {2005: 2, 2006: 2, 2007: 2, 2008: 2, 2009: 2, 2010: 2,
                2011: 3, 2012: 2, 2013: 2, 2014: 3, 2015: 2}

roubos_por_ano = mensal.groupby("ano")["total_roubos"].sum()
ano_pico_reg = roubos_por_ano.idxmax()
ano_vale_reg = roubos_por_ano.loc[ano_pico_reg:].idxmin()

reg = mensal[["data", "ano", "mes"] + COLS_PRINCIPAIS].copy()
reg["t"] = np.arange(len(reg))                       # 0 = primeiro mês da série
reg["carnaval"] = (reg["mes"] == reg["ano"].map(MES_CARNAVAL)).astype(int)

# Variáveis de "quebra": valem 0 até dezembro do ano da quebra e depois crescem
# 1, 2, 3... Com elas, a reta pode mudar de inclinação nesses pontos.
t_pico = reg.loc[(reg["ano"] == ano_pico_reg) & (reg["mes"] == 12), "t"].iloc[0]
t_vale = reg.loc[(reg["ano"] == ano_vale_reg) & (reg["mes"] == 12), "t"].iloc[0]
reg["apos_pico"] = np.maximum(0, reg["t"] - t_pico)
reg["apos_vale"] = np.maximum(0, reg["t"] - t_vale)


def ajustar(formula):
    """Regressão por MQO com erros padrão robustos (HAC, 12 meses)."""
    return smf.ols(formula, data=reg).fit(cov_type="HAC", cov_kwds={"maxlags": 12})


# ------------------------------------------------------------
# 11.3 Modelo 1 - tendência simples (uma reta)
# ------------------------------------------------------------
# y = b0 + b1*t  ->  b1 é a variação média de ocorrências a cada mês.
m_furtos_reta = ajustar("total_furtos ~ t")
m_roubos_reta = ajustar("total_roubos ~ t")

# ------------------------------------------------------------
# 11.4 Modelo 2 - roubos em três fases (regressão segmentada)
# ------------------------------------------------------------
# As quebras vêm do pico e do vale anuais, ou seja, foram escolhidas olhando
# os mesmos dados. Os p-valores deste modelo são, portanto, otimistas.
m_roubos_fases = ajustar("total_roubos ~ t + apos_pico + apos_vale")


def inclinacao(termos, fase):
    """Inclinação de uma fase = soma dos coeficientes, com IC de 95%."""
    teste = m_roubos_fases.t_test(" + ".join(termos) + " = 0")
    inf, sup = np.squeeze(teste.conf_int())
    return {"fase": fase, "roubos_por_mes": float(np.squeeze(teste.effect)),
            "ic95_inf": float(inf), "ic95_sup": float(sup)}


fases_roubos = pd.DataFrame([
    inclinacao(["t"], f"{ANOS[0]}–{ano_pico_reg}"),
    inclinacao(["t", "apos_pico"], f"{ano_pico_reg + 1}–{ano_vale_reg}"),
    inclinacao(["t", "apos_pico", "apos_vale"], f"{ano_vale_reg + 1}–{ANOS[-1]}"),
]).round(1)

# ------------------------------------------------------------
# 11.5 Modelo 3 - furtos: regressão múltipla com Carnaval
# ------------------------------------------------------------
# t e t² permitem uma tendência curva (sobe e depois achata);
# carnaval vale 1 no mês do Carnaval e 0 nos demais (variável dummy).
# O summary() avisa de "condition number" alto porque t e t² andam juntos.
# Isso é esperado aqui e não afeta os valores ajustados nem o efeito do Carnaval.
m_furtos_mult = ajustar("total_furtos ~ t + I(t ** 2) + carnaval")

efeito_carnaval = m_furtos_mult.params["carnaval"]
ic_carnaval = m_furtos_mult.conf_int().loc["carnaval"]
t_topo_furtos = -m_furtos_mult.params["t"] / (2 * m_furtos_mult.params["I(t ** 2)"])
data_topo_furtos = reg.loc[int(round(t_topo_furtos)), "data"]

# ------------------------------------------------------------
# 11.6 Comparação e diagnóstico dos modelos
# ------------------------------------------------------------
# Durbin-Watson: perto de 2 = pouca autocorrelação; perto de 0 = muita.
# Breusch-Pagan: p baixo indica variância dos erros não constante.
# Jarque-Bera: p baixo indica resíduos distantes da distribuição normal.
MODELOS = {
    "1a. Furtos: reta":          m_furtos_reta,
    "1b. Roubos: reta":          m_roubos_reta,
    "2. Roubos: três fases":     m_roubos_fases,
    "3. Furtos: curva+Carnaval": m_furtos_mult,
}

resumo_modelos = pd.DataFrame([{
    "modelo":            nome,
    "n":                 int(m.nobs),
    "r2":                m.rsquared,
    "r2_ajustado":       m.rsquared_adj,
    "erro_medio":        np.sqrt(m.mse_resid),        # em ocorrências por mês
    "durbin_watson":     durbin_watson(m.resid),
    "p_breusch_pagan":   het_breuschpagan(m.resid, m.model.exog)[1],
    "p_jarque_bera":     jarque_bera(m.resid)[1],
} for nome, m in MODELOS.items()]).round(4).set_index("modelo")

coeficientes = pd.concat([
    pd.DataFrame({
        "modelo":          nome,
        "termo":           m.params.index,
        "coeficiente":     m.params.values,
        "erro_padrao_hac": m.bse.values,
        "p_valor":         m.pvalues.values,
        "ic95_inf":        m.conf_int()[0].values,
        "ic95_sup":        m.conf_int()[1].values,
    }) for nome, m in MODELOS.items()
], ignore_index=True).round(4)

print("\n=== Regressão linear: comparação dos modelos ===")
display(resumo_modelos)
print("\n=== Regressão linear: coeficientes (erros padrão HAC) ===")
display(coeficientes)
print("\n=== Modelo 2: inclinação dos roubos em cada fase (ocorrências por mês) ===")
display(fases_roubos)
print(f"\nCarnaval: {efeito_carnaval:+,.0f} furtos no mês "
      f"(IC 95%: {ic_carnaval[0]:,.0f} a {ic_carnaval[1]:,.0f})")
print(f"Topo da tendência dos furtos: {data_topo_furtos:%m/%Y}")

print("\n=== Relatório completo do modelo 3 ===")
print(m_furtos_mult.summary())

# Valores ajustados, usados nos gráficos abaixo e nos slides
for nome, modelo in [("furtos_reta", m_furtos_reta), ("roubos_reta", m_roubos_reta),
                     ("roubos_fases", m_roubos_fases), ("furtos_mult", m_furtos_mult)]:
    reg[f"ajuste_{nome}"] = modelo.fittedvalues
    reg[f"residuo_{nome}"] = modelo.resid

# Observado x ajustado
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, facecolor=COR_FUNDO)
paineis = [
    (axes[0, 0], "total_roubos", "roubos_reta",  COR_SERIE,   "Modelo 1b: roubos, uma reta"),
    (axes[0, 1], "total_roubos", "roubos_fases", COR_SERIE,   "Modelo 2: roubos, três fases"),
    (axes[1, 0], "total_furtos", "furtos_reta",  COR_FURTO_G, "Modelo 1a: furtos, uma reta"),
    (axes[1, 1], "total_furtos", "furtos_mult",  COR_FURTO_G, "Modelo 3: furtos, curva + Carnaval"),
]
for ax, y, ajuste, cor, titulo in paineis:
    r2_painel = 1 - reg[f"residuo_{ajuste}"].var() / reg[y].var()
    ax.plot(reg["data"], reg[y], color=cor, linewidth=1.2, alpha=0.75, label="observado")
    ax.plot(reg["data"], reg[f"ajuste_{ajuste}"], color="#0b0b0b", linewidth=2, label="modelo")
    estilo_eixo(ax, f"{titulo} (R² = {r2_painel:.2f})", "ocorrências no mês")
    ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig("regressao_observado_ajustado.png", dpi=150)
plt.show()

# Resíduos: sobra sem padrão visível é sinal de bom ajuste
fig, ax = plt.subplots(figsize=(12, 3.5), facecolor=COR_FUNDO)
ax.bar(reg["data"], reg["residuo_furtos_mult"], width=25, color=COR_FURTO_G)
ax.axhline(0, color=COR_TEXTO, linewidth=1)
estilo_eixo(ax, "Resíduos do modelo 3 (observado − modelo)", "ocorrências no mês")
fig.tight_layout()
fig.savefig("regressao_residuos.png", dpi=150)
plt.show()

# ------------------------------------------------------------
# 12. Salvar tudo em uma planilha Excel única (uma aba por tabela)
# ------------------------------------------------------------
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as ImagemExcel

ARQUIVO_EXCEL = "projeto_seguranca_rj_2005_2015.xlsx"

# Cada aba: (tabela, escrever_indice?). Nomes de aba têm no máximo 31 caracteres.
abas = {
    "Base CISP x mes":         (df_rj, False),
    "Estat CISP x mes":        (stats_cisp.rename_axis("variavel"), True),
    "Estat municipio x mes":   (stats_mensal.rename_axis("variavel"), True),
    "Estat por ano":           (stats_por_ano, True),
    "Taxa por 100 mil":        (taxas, True),
    "Resumo outliers":         (resumo_outliers.rename_axis("variavel"), True),
    "Outliers por CISP":       (detalhe_outliers, False),
    "Outliers municipio":      (outliers_municipio, False),
    "Composicao roubos":       (composicao_roubos, True),
    "Composicao furtos":       (composicao_furtos, True),
    "Por delegacia":           (por_cisp, False),
    "Correlacao":              (correlacao.rename_axis("variavel"), True),
    "Distribuicao mensal":     (distribuicao, True),
    "Regressao modelos":       (resumo_modelos, True),
    "Regressao coeficientes":  (coeficientes, False),
    "Regressao fases roubos":  (fases_roubos, False),
}

leia_me = pd.DataFrame([
    ("Base CISP x mes",       "Base processada: uma linha por delegacia (CISP) e mês, Rio de Janeiro, 2005-2015. "
                              "Células vazias em bicicleta = dado não coletado antes de 2014."),
    ("Estat CISP x mes",      "Média, mediana, desvio padrão (amostral), CV, quartis, IQR e assimetria de cada "
                              "variável, considerando cada CISP x mês como uma observação."),
    ("Estat municipio x mes", "As mesmas medidas sobre o total mensal do município (soma das CISPs, 132 meses)."),
    ("Estat por ano",         "Média, mediana e desvio padrão dos totais mensais de roubos e furtos, ano a ano."),
    ("Taxa por 100 mil",      "Total anual de roubos e furtos do município dividido pela população (IBGE) × 100.000. "
                              "Mostra se o crime cresceu mais do que a cidade."),
    ("Resumo outliers",       "Quantidade e % de outliers por variável em três critérios: IQR global (1,5×IQR), "
                              "z-score > 3 e IQR calculado dentro de cada CISP."),
    ("Outliers por CISP",     "Meses atípicos de cada delegacia em total_roubos e total_furtos (IQR dentro da CISP), "
                              "com a mediana da própria CISP para comparação."),
    ("Outliers municipio",    "Meses atípicos do município como um todo, com os limites do IQR e o resultado de cada método."),
    ("Composicao roubos",     "Total e participação de cada tipo de roubo. roubo_rua já inclui transeunte, celular e coletivo."),
    ("Composicao furtos",     "Total e participação de cada tipo de furto."),
    ("Por delegacia",         "Roubos e furtos de cada CISP no período, com a participação no total da cidade."),
    ("Correlacao",            "Correlação de Pearson entre os totais mensais das principais variáveis."),
    ("Distribuicao mensal",   "Quantidade de meses em cada faixa de 500 ocorrências (dados dos histogramas)."),
    ("Regressao modelos",     "Comparação dos modelos: R², R² ajustado, erro médio e testes de Durbin-Watson, "
                              "Breusch-Pagan e Jarque-Bera."),
    ("Regressao coeficientes", "Coeficientes, erros padrão HAC (Newey-West), p-valores e IC de 95% de cada modelo."),
    ("Regressao fases roubos", "Variação média de roubos por mês em cada fase do modelo segmentado."),
    ("Grafico boxplot",       "Boxplot dos totais mensais de roubos e furtos por ano."),
    ("Grafico regressao",     "Valores observados e ajustados dos modelos de regressão."),
    ("Fonte",                 "ISP-RJ, BaseDPEvolucaoMensalCisp.csv (https://www.ispdados.rj.gov.br/). "
                              "População: IBGE/SIDRA (tabelas 6579, 793 e 202). "
                              "Os dados ficam no banco SQLite seguranca_rj.db; totais mensais e taxas saem de "
                              "consultas SQL e as demais estatísticas foram calculadas em Python (pandas)."),
], columns=["aba", "conteudo"])

FONTE        = Font(name="Arial", size=10)
FONTE_CAB    = Font(name="Arial", size=10, bold=True, color="FFFFFF")
FUNDO_CAB    = PatternFill("solid", fgColor="2A78D6")


def formatar_aba(ws, tabela, com_indice):
    """Fonte Arial, cabeçalho destacado, filtro, painel congelado e larguras."""
    for linha in ws.iter_rows():
        for celula in linha:
            celula.font = FONTE
    for celula in ws[1]:
        celula.font = FONTE_CAB
        celula.fill = FUNDO_CAB
        celula.alignment = Alignment(horizontal="center", vertical="center")

    # Colunas decimais com separador de milhar; inteiros (ano, mês, cisp) ficam como estão
    colunas = ([tabela.index.name] if com_indice else []) + list(tabela.columns)
    decimais = set(tabela.select_dtypes("float").columns)
    for i, nome in enumerate(colunas, start=1):
        letra = get_column_letter(i)
        if nome in decimais:
            for celula in ws[letra][1:]:
                celula.number_format = "#,##0.00"
        amostra = [str(nome)] + [str(c.value) for c in ws[letra][1:200] if c.value is not None]
        ws.column_dimensions[letra].width = min(max(map(len, amostra)) + 3, 45)

    ws.freeze_panes = "B2" if com_indice else "A2"
    ws.auto_filter.ref = ws.dimensions


with pd.ExcelWriter(ARQUIVO_EXCEL, engine="openpyxl") as writer:
    leia_me.to_excel(writer, sheet_name="Leia-me", index=False)
    for nome_aba, (tabela, com_indice) in abas.items():
        tabela.to_excel(writer, sheet_name=nome_aba, index=com_indice)

    wb = writer.book
    for nome_aba, (tabela, com_indice) in abas.items():
        formatar_aba(wb[nome_aba], tabela, com_indice)

    # Aba Leia-me: texto longo com quebra de linha
    ws = wb["Leia-me"]
    formatar_aba(ws, leia_me, False)
    ws.column_dimensions["A"].width = 24
    ws.column_dimensions["B"].width = 110
    for celula in ws["B"][1:]:
        celula.alignment = Alignment(wrap_text=True, vertical="top")

    # Aba com o gráfico gerado no item 11
    ws = wb.create_sheet("Grafico boxplot")
    imagem = ImagemExcel("boxplot_roubos_furtos_por_ano.png")
    imagem.width, imagem.height = imagem.width * 0.6, imagem.height * 0.6
    ws.add_image(imagem, "A1")

    # Aba com o gráfico gerado no item 11.6
    ws = wb.create_sheet("Grafico regressao")
    imagem = ImagemExcel("regressao_observado_ajustado.png")
    imagem.width, imagem.height = imagem.width * 0.6, imagem.height * 0.6
    ws.add_image(imagem, "A1")

print("\nPlanilha salva:", ARQUIVO_EXCEL)
print("Abas:", ", ".join(["Leia-me", *abas, "Grafico boxplot", "Grafico regressao"]))

# files.download(ARQUIVO_EXCEL)  # descomente se quiser baixar

# ------------------------------------------------------------
# 12.1 Guardar os resultados também no banco
# ------------------------------------------------------------
# Assim o .db fica completo: base + estatísticas + outliers + regressão, consultáveis por SQL.
resultados_db = {
    "estat_cisp_mes":       stats_cisp.rename_axis("variavel").reset_index(),
    "estat_municipio_mes":  stats_mensal.rename_axis("variavel").reset_index(),
    "estat_por_ano":        stats_por_ano.reset_index(),
    "resumo_outliers":      resumo_outliers.rename_axis("variavel").reset_index(),
    "outliers_por_cisp":    detalhe_outliers,
    "outliers_municipio":   outliers_municipio,
    "composicao_roubos":    composicao_roubos.reset_index(),
    "composicao_furtos":    composicao_furtos.reset_index(),
    "por_delegacia":        por_cisp,
    "regressao_modelos":    resumo_modelos.reset_index(),
    "regressao_coeficientes": coeficientes,
    "regressao_fases_roubos": fases_roubos,
}
for nome_tabela, tabela in resultados_db.items():
    tabela.to_sql(nome_tabela, conn, if_exists="replace", index=False)
conn.commit()

tabelas_db = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", conn)
print(f"\nTabelas no banco {ARQUIVO_DB}:", ", ".join(tabelas_db["name"]))
conn.close()

files.download(ARQUIVO_DB)  # descomente se quiser baixar



# ============================================================
# ETAPA 3 - APRESENTAÇÃO DE CONCLUSÕES (PowerPoint)
# ============================================================
# Todos os números dos slides são calculados aqui a partir das tabelas
# acima; se a base mudar, a apresentação acompanha.

try:
    import pptx  # noqa: F401
except ImportError:                       # o Colab não traz o python-pptx
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"], check=True)

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import (XL_CHART_TYPE, XL_LEGEND_POSITION, XL_MARKER_STYLE, XL_LABEL_POSITION,
                             XL_TICK_LABEL_POSITION)
from pptx.oxml.ns import qn
from lxml import etree

ARQUIVO_PPTX = "apresentacao_seguranca_rj_2005_2015.pptx"

# ------------------------------------------------------------
# 13. Números usados nos slides
# ------------------------------------------------------------
MESES = ["jan", "fev", "mar", "abr", "mai", "jun",
         "jul", "ago", "set", "out", "nov", "dez"]
MESES_EXTENSO = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
                 "agosto", "setembro", "outubro", "novembro", "dezembro"]
NOMES_DP = {5: "Centro", 12: "Copacabana", 14: "Leblon", 16: "Barra da Tijuca", 35: "Campo Grande"}


def num(valor):
    """Número no padrão brasileiro: 89193 -> '89.193'."""
    return f"{valor:,.0f}".replace(",", ".")


def pct(fracao):
    """Variação percentual com sinal: -0.386 -> '−39%'."""
    return f"{fracao * 100:+.0f}%".replace("-", "−")


anual = mensal.groupby("ano")[COLS_PRINCIPAIS].sum()
ano_ini, ano_fim = anual.index.min(), anual.index.max()

# Pergunta 1 - roubos: pico, vale e retomada
ano_pico_roubo = anual["total_roubos"].idxmax()
ano_vale_roubo = anual.loc[ano_pico_roubo:, "total_roubos"].idxmin()
queda_roubos = anual.loc[ano_vale_roubo, "total_roubos"] / anual.loc[ano_pico_roubo, "total_roubos"] - 1
retomada_roubos = anual.loc[ano_fim, "total_roubos"] / anual.loc[ano_vale_roubo, "total_roubos"] - 1

# Pergunta 2 - sazonalidade dos furtos
media_mes_furtos = mensal.groupby("mes")["total_furtos"].mean()
mes_pico_furto = media_mes_furtos.idxmax()
acima_media = media_mes_furtos[mes_pico_furto] / media_mes_furtos.drop(mes_pico_furto).mean() - 1

furtos_cisp = detalhe_outliers.query("variavel == 'total_furtos'")
cisp_destaque = furtos_cisp["cisp"].value_counts().idxmax()
nome_cisp = f"{cisp_destaque}ª DP" + (f" ({NOMES_DP[cisp_destaque]})" if cisp_destaque in NOMES_DP else "")
mediana_mes_cisp = df_rj.loc[df_rj["cisp"] == cisp_destaque].groupby("mes")["total_furtos"].median()
pico_cisp = mediana_mes_cisp[mes_pico_furto]
resto_cisp = mediana_mes_cisp.drop(mes_pico_furto).median()

# Pergunta 3 - furtos x população
cresc_furtos = anual.loc[ano_fim, "total_furtos"] / anual.loc[ano_ini, "total_furtos"] - 1
cresc_pop = POPULACAO_RIO[ano_fim] / POPULACAO_RIO[ano_ini] - 1
col_taxa_furto, col_taxa_roubo = "taxa_total_furtos_100mil", "taxa_total_roubos_100mil"
cresc_taxa_furtos = taxas.loc[ano_fim, col_taxa_furto] / taxas.loc[ano_ini, col_taxa_furto] - 1

furtos_maior = anual["total_furtos"] > anual["total_roubos"]
ano_virada = next((a for a in anual.index if furtos_maior.loc[a:].all()), None)

# Outliers do município
furtos_mun = outliers_municipio.query("variavel == 'total_furtos' and outlier_iqr == 'sim'")
altos = furtos_mun[furtos_mun["valor"] > furtos_mun["limite_superior_iqr"]]
baixos = furtos_mun[furtos_mun["valor"] < furtos_mun["limite_inferior_iqr"]]
altos_rotulos = [f"{MESES[m - 1]}/{a}" for a, m in zip(altos["ano"], altos["mes"])]
altos_carnaval = len(altos) > 0 and altos["mes"].isin([2, 3]).all()

def dec(valor, casas=2):
    """Decimal no padrão brasileiro: 1660.08 -> '1.660,08'."""
    return f"{valor:,.{casas}f}".replace(",", "#").replace(".", ",").replace("#", ".").replace("-", "−")


def sinal(valor):
    """Inteiro com sinal: -53.4 -> '−53'."""
    return f"{valor:+,.0f}".replace(",", ".").replace("-", "−")


def p_txt(p):
    return "< 0,001" if p < 0.001 else dec(p, 3)


# Análise descritiva
est_r, est_f = stats_mensal.loc["total_roubos"], stats_mensal.loc["total_furtos"]
parte_rua = composicao_roubos.loc["roubo_rua", "participacao_%"]
parte_outros_furtos = composicao_furtos.loc["outros_furtos", "participacao_%"]
top5_furtos, top5_roubos = parcela_maiores("furtos", 5), parcela_maiores("roubos", 5)
corr_rf = correlacao.loc["total_roubos", "total_furtos"]
mediana_furtos_mil = round(est_f["mediana"] / 1000)

# Regressão
R2_FURTOS_RETA, R2_ROUBOS_RETA = m_furtos_reta.rsquared, m_roubos_reta.rsquared
R2_ROUBOS_FASES, R2_FURTOS_MULT = m_roubos_fases.rsquared, m_furtos_mult.rsquared
inc_furtos_reta = m_furtos_reta.params["t"]
inc_roubos_reta = m_roubos_reta.params["t"]
p_roubos_reta = m_roubos_reta.pvalues["t"]
topo_furtos_txt = f"{MESES[data_topo_furtos.month - 1]}/{data_topo_furtos.year}"

# ------------------------------------------------------------
# 14. Estilo e funções auxiliares dos slides
# ------------------------------------------------------------
COR_ESCURO  = RGBColor(0x1C, 0x23, 0x33)   # fundo da capa e da conclusão
COR_TEXTO_S = RGBColor(0x1C, 0x23, 0x33)
COR_MUTED   = RGBColor(0x5B, 0x61, 0x70)
COR_CLARO   = RGBColor(0xF2, 0xF4, 0xF7)   # fundo dos cartões
COR_BRANCO  = RGBColor(0xFF, 0xFF, 0xFF)
COR_ROUBO   = RGBColor(0x2A, 0x78, 0xD6)   # mesma cor do boxplot
COR_FURTO   = RGBColor(0xEB, 0x68, 0x34)
COR_SUAVE   = RGBColor(0xC9, 0xCE, 0xD6)
COR_GRADE_S = RGBColor(0xE3, 0xE6, 0xEB)
FONTE_TITULO, FONTE_CORPO = "Cambria", "Calibri"

prs = Presentation()
prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
LAYOUT_VAZIO = prs.slide_layouts[6]


def novo_slide(fundo=COR_BRANCO):
    slide = prs.slides.add_slide(LAYOUT_VAZIO)
    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = fundo
    return slide


def texto(slide, x, y, w, h, linhas, tamanho=16, cor=COR_TEXTO_S, negrito=False,
          fonte=FONTE_CORPO, alinhar=PP_ALIGN.LEFT, ancora=MSO_ANCHOR.TOP, espaco=6):
    """Caixa de texto sem margem interna; `linhas` é uma string ou lista de strings."""
    caixa = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = caixa.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
    tf.vertical_anchor = ancora
    for i, linha in enumerate([linhas] if isinstance(linhas, str) else linhas):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinhar
        p.space_after = Pt(espaco)
        run = p.add_run()
        run.text = linha
        run.font.size, run.font.bold, run.font.name = Pt(tamanho), negrito, fonte
        run.font.color.rgb = cor
    return caixa


def cabecalho(slide, rotulo, titulo):
    texto(slide, 0.6, 0.45, 12, 0.3, rotulo.upper(), 12, COR_FURTO, negrito=True)
    texto(slide, 0.6, 0.75, 12.1, 0.9, titulo, 30, COR_TEXTO_S, negrito=True, fonte=FONTE_TITULO)


def cartao(slide, x, y, w, h, cor=COR_CLARO):
    forma = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(x), Inches(y), Inches(w), Inches(h))
    forma.adjustments[0] = 0.06
    forma.fill.solid()
    forma.fill.fore_color.rgb = cor
    forma.line.fill.background()
    forma.shadow.inherit = False
    return forma


def circulo(slide, x, y, d, rotulo, cor=COR_FURTO, tamanho=18, fonte=FONTE_CORPO):
    forma = slide.shapes.add_shape(MSO_SHAPE.OVAL, Inches(x), Inches(y), Inches(d), Inches(d))
    forma.fill.solid()
    forma.fill.fore_color.rgb = cor
    forma.line.fill.background()
    forma.shadow.inherit = False
    tf = forma.text_frame
    tf.margin_left = tf.margin_right = tf.margin_top = tf.margin_bottom = 0
    tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.CENTER
    run = p.add_run()
    run.text = rotulo
    run.font.size, run.font.bold, run.font.name = Pt(tamanho), True, fonte
    run.font.color.rgb = COR_BRANCO


def destaque(slide, x, y, w, numero, legenda, cor=COR_FURTO, tamanho=54):
    """Número grande com legenda curta embaixo."""
    texto(slide, x, y, w, 0.95, numero, tamanho, cor, negrito=True, fonte=FONTE_TITULO)
    texto(slide, x, y + 1.0, w, 0.6, legenda, 14, COR_MUTED)


def notas(slide, conteudo):
    slide.notes_slide.notes_text_frame.text = conteudo


def estilo_grafico(grafico, legenda=True):
    grafico.font.name, grafico.font.size = FONTE_CORPO, Pt(12)
    grafico.font.color.rgb = COR_MUTED
    grafico.has_title = False
    grafico.has_legend = legenda
    if legenda:
        grafico.legend.position = XL_LEGEND_POSITION.TOP
        grafico.legend.include_in_layout = False
    eixo_v, eixo_c = grafico.value_axis, grafico.category_axis
    eixo_v.has_major_gridlines = True
    eixo_v.major_gridlines.format.line.color.rgb = COR_GRADE_S
    eixo_v.format.line.fill.background()
    eixo_v.tick_labels.number_format = '#,##0'
    eixo_v.tick_labels.number_format_is_linked = False
    eixo_c.format.line.color.rgb = COR_SUAVE
    eixo_c.has_major_gridlines = False


def pular_rotulos(grafico, a_cada):
    """Mostra só 1 rótulo do eixo X a cada `a_cada` categorias (ex.: 1 por ano)."""
    cat_ax = grafico._chartSpace.find(".//" + qn("c:catAx"))
    fim = cat_ax.find(qn("c:noMultiLvlLbl"))   # o esquema exige as tags antes desta
    for tag in ("c:tickLblSkip", "c:tickMarkSkip"):
        el = etree.Element(qn(tag))
        el.set("val", str(a_cada))
        if fim is not None:
            fim.addprevious(el)
        else:
            cat_ax.append(el)


def colunas(slide, x, y, w, h, categorias, valores, nome, cor, destaque_idx=None):
    dados = CategoryChartData()
    dados.categories = categorias
    dados.add_series(nome, valores)
    grafico = slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(x), Inches(y),
                                     Inches(w), Inches(h), dados).chart
    estilo_grafico(grafico, legenda=False)
    plot = grafico.plots[0]
    plot.gap_width = 60
    serie = plot.series[0]
    serie.invert_if_negative = False
    serie.format.fill.solid()
    serie.format.fill.fore_color.rgb = cor if destaque_idx is None else COR_SUAVE
    if destaque_idx is not None:
        ponto = serie.points[destaque_idx]
        ponto.format.fill.solid()
        ponto.format.fill.fore_color.rgb = cor
    plot.has_data_labels = True
    plot.data_labels.number_format = '#,##0'
    plot.data_labels.number_format_is_linked = False
    plot.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END
    plot.data_labels.font.size = Pt(11)
    return grafico


def linhas_grafico(slide, x, y, w, h, categorias, series, marcadores=False):
    dados = CategoryChartData()
    dados.categories = categorias
    for nome, valores, _ in series:
        dados.add_series(nome, valores)
    grafico = slide.shapes.add_chart(XL_CHART_TYPE.LINE, Inches(x), Inches(y),
                                     Inches(w), Inches(h), dados).chart
    estilo_grafico(grafico)
    for serie, (_, _, cor) in zip(grafico.plots[0].series, series):
        serie.smooth = False
        serie.format.line.color.rgb = cor
        serie.format.line.width = Pt(2.25)
        if marcadores:
            serie.marker.style = XL_MARKER_STYLE.CIRCLE
            serie.marker.size = 8
            serie.marker.format.fill.solid()
            serie.marker.format.fill.fore_color.rgb = cor
            serie.marker.format.line.color.rgb = COR_BRANCO
        else:
            serie.marker.style = XL_MARKER_STYLE.NONE
    return grafico


def tabela_slide(slide, x, y, w, linhas, larguras, altura=0.44, tamanho=14):
    """Tabela nativa: 1ª linha é o cabeçalho; colunas após a 1ª ficam à direita."""
    forma = slide.shapes.add_table(len(linhas), len(linhas[0]), Inches(x), Inches(y),
                                   Inches(w), Inches(altura * len(linhas)))
    tabela = forma.table
    for j, proporcao in enumerate(larguras):
        tabela.columns[j].width = Inches(w * proporcao)
    for i, linha in enumerate(linhas):
        tabela.rows[i].height = Inches(altura)
        for j, valor in enumerate(linha):
            celula = tabela.cell(i, j)
            celula.fill.solid()
            celula.fill.fore_color.rgb = COR_ESCURO if i == 0 else (COR_CLARO if i % 2 == 0 else COR_BRANCO)
            celula.vertical_anchor = MSO_ANCHOR.MIDDLE
            p = celula.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.LEFT if j == 0 else PP_ALIGN.RIGHT
            run = p.add_run()
            run.text = str(valor)
            run.font.size, run.font.name, run.font.bold = Pt(tamanho), FONTE_CORPO, i == 0
            run.font.color.rgb = COR_BRANCO if i == 0 else COR_TEXTO_S
    return tabela


def barras(slide, x, y, w, h, categorias, valores, nome, cor, formato='0"%"'):
    """Barras horizontais, maior valor no topo, com o valor escrito em cada barra."""
    dados_g = CategoryChartData()
    dados_g.categories = categorias
    dados_g.add_series(nome, valores)
    grafico = slide.shapes.add_chart(XL_CHART_TYPE.BAR_CLUSTERED, Inches(x), Inches(y),
                                     Inches(w), Inches(h), dados_g).chart
    estilo_grafico(grafico, legenda=False)
    grafico.category_axis.reverse_order = True
    grafico.value_axis.visible = False
    grafico.value_axis.has_major_gridlines = False
    plot = grafico.plots[0]
    plot.gap_width = 45
    plot.series[0].format.fill.solid()
    plot.series[0].format.fill.fore_color.rgb = cor
    plot.has_data_labels = True
    plot.data_labels.number_format = formato
    plot.data_labels.number_format_is_linked = False
    plot.data_labels.position = XL_LABEL_POSITION.OUTSIDE_END
    plot.data_labels.font.size = Pt(12)
    return grafico


def codigo(slide, x, y, w, h, linhas, tamanho=15):
    """Trecho de código em fonte monoespaçada sobre fundo escuro."""
    cartao(slide, x, y, w, h, COR_ESCURO)
    texto(slide, x + 0.3, y + 0.25, w - 0.6, h - 0.5, linhas, tamanho, COR_BRANCO,
          fonte="Consolas", espaco=3)


# ------------------------------------------------------------
# 15. Slides
# ------------------------------------------------------------
anos_txt = [str(a) for a in anual.index]
mes_pico_nome = MESES_EXTENSO[mes_pico_furto - 1]

# --- 1. Capa ---------------------------------------------------
s = novo_slide(COR_ESCURO)
texto(s, 0.9, 1.3, 11, 0.4, "PROJETO BIG DATA · SEGURANÇA PÚBLICA", 14, COR_FURTO, negrito=True)
texto(s, 0.9, 1.9, 11.5, 2.0, "Roubos e furtos no Rio de Janeiro", 50, COR_BRANCO,
      negrito=True, fonte=FONTE_TITULO)
texto(s, 0.9, 3.95, 11, 0.9, f"O que {ano_fim - ano_ini + 1} anos de registros policiais mostram "
      f"({ano_ini}–{ano_fim})", 24, COR_SUAVE)
texto(s, 0.9, 6.3, 11, 0.4, "Fonte dos dados: Instituto de Segurança Pública (ISP-RJ) e IBGE", 13, COR_SUAVE)
notas(s, "Apresentar o tema: a evolução dos roubos e furtos registrados na cidade do Rio "
         f"de Janeiro entre {ano_ini} e {ano_fim}, com dados públicos do ISP.")

# --- 2. Dados e pipeline -----------------------------------------
s = novo_slide()
cabecalho(s, "Os dados", "De onde vêm os números")
destaque(s, 0.6, 1.95, 3.6, num(df_rj["cisp"].nunique()), "delegacias (CISPs) na cidade", COR_ROUBO)
destaque(s, 0.6, 3.55, 3.6, num(len(mensal)), "meses analisados", COR_ROUBO)
destaque(s, 0.6, 5.15, 3.6, num(len(df_rj)), "registros (delegacia × mês)", COR_ROUBO)
etapas = [
    ("Coleta", "Base mensal por delegacia, baixada do portal do ISP-RJ"),
    ("Filtro", f"Só o município do Rio, de {ano_ini} a {ano_fim}"),
    ("Limpeza", "Tipos numéricos; dados não coletados ficam vazios, não zero"),
    ("Banco de dados", "Carga em SQLite (delegacias, ocorrências, população) e consultas SQL"),
    ("Estatística", "Análise descritiva, detecção de outliers e regressão linear"),
    ("Resultado", "Planilha Excel única e esta apresentação, gerados pelo código"),
]
for i, (nome, desc) in enumerate(etapas):
    y = 1.95 + i * 0.85
    circulo(s, 4.9, y, 0.6, str(i + 1), COR_ESCURO, 16)
    texto(s, 5.75, y - 0.02, 6.9, 0.35, nome, 18, COR_TEXTO_S, negrito=True)
    texto(s, 5.75, y + 0.34, 6.9, 0.4, desc, 14, COR_MUTED)
notas(s, "Cada linha da base é uma delegacia em um mês. O pipeline em Python baixa, filtra "
         "e limpa os dados, carrega tudo num banco SQLite e consulta o banco com SQL para "
         "calcular as estatísticas; a planilha e esta apresentação saem do mesmo código.")

# --- 3. Perguntas ------------------------------------------------
s = novo_slide()
cabecalho(s, "Objetivo", "Quatro perguntas que os dados respondem")
perguntas = [
    ("Os roubos caíram?", f"Como os roubos evoluíram de {ano_ini} a {ano_fim}, e a queda se manteve?"),
    ("Existe época do ano?", "Algum mês concentra mais furtos do que os outros?"),
    ("Mais crime ou mais gente?", "Os furtos cresceram mais do que a população da cidade?"),
    ("Dá para medir?", "Quanto mudam por mês, e quanto pesa o Carnaval? A regressão responde."),
]
for i, (titulo_p, desc) in enumerate(perguntas):
    x = 0.6 + i * 3.08
    cartao(s, x, 2.1, 2.85, 4.1)
    circulo(s, x + 0.3, 2.45, 0.75, str(i + 1), COR_FURTO, 22)
    texto(s, x + 0.3, 3.4, 2.3, 1.0, titulo_p, 20, COR_TEXTO_S, negrito=True, fonte=FONTE_TITULO)
    texto(s, x + 0.3, 4.45, 2.3, 1.6, desc, 15, COR_MUTED)
notas(s, "Em vez de apresentar todas as estatísticas, o trabalho se organiza em torno de "
         "quatro perguntas. Antes, uma análise descritiva mostra o retrato geral dos dados.")

# --- 4. Série mensal --------------------------------------------
s = novo_slide()
titulo_serie = (f"Desde {ano_virada}, a cidade registra mais furtos do que roubos"
                if ano_virada else "Roubos e furtos registrados, mês a mês")
cabecalho(s, "Visão geral", titulo_serie)
g = linhas_grafico(
    s, 0.6, 1.8, 12.1, 4.9,
    [str(a) for a in mensal["ano"]],
    [("Roubos", mensal["total_roubos"].tolist(), COR_ROUBO),
     ("Furtos", mensal["total_furtos"].tolist(), COR_FURTO)],
)
pular_rotulos(g, 12)
texto(s, 0.6, 6.8, 12.1, 0.4, "Total de ocorrências registradas por mês no município do Rio de Janeiro. "
      "Fonte: ISP-RJ.", 11, COR_MUTED)
notas(s, "Cada ponto é um mês. Repare em três coisas: a queda dos roubos no meio do período, "
         "a subida dos furtos e os picos de furtos que se repetem todo começo de ano.")

# --- 5. Análise descritiva: resumo --------------------------------
s = novo_slide()
cabecalho(s, "Análise descritiva · Resumo",
          f"Em um mês típico: cerca de {num(est_r['media'])} roubos e {num(est_f['media'])} furtos")
medidas = [("Média", "media", 0), ("Mediana", "mediana", 0), ("Desvio padrão", "desvio_padrao", 0),
           ("Mínimo", "minimo", 0), ("Máximo", "maximo", 0)]
linhas_tabela = [["Total mensal da cidade", "Roubos", "Furtos"]]
linhas_tabela += [[nome, num(est_r[chave]), num(est_f[chave])] for nome, chave, _ in medidas]
linhas_tabela += [["Coef. de variação", f"{dec(est_r['cv_%'], 1)}%", f"{dec(est_f['cv_%'], 1)}%"],
                  ["Assimetria", dec(est_r["assimetria"]), dec(est_f["assimetria"])]]
tabela_slide(s, 0.6, 2.0, 6.6, linhas_tabela, [0.5, 0.25, 0.25], altura=0.55, tamanho=16)
leituras = [
    ("Média ≈ mediana", "Na cidade, os totais mensais são quase simétricos: a média representa bem um mês comum."),
    (f"Variação de cerca de {dec(est_f['cv_%'], 0)}%", "O desvio padrão é pequeno perto da média, mesmo com "
                                                      f"{len(mensal)} meses de altos e baixos."),
    ("Por delegacia é diferente", f"Em cada delegacia, furtos têm assimetria {dec(stats_cisp.loc['total_furtos', 'assimetria'])} "
                                  f"e máximo de {num(stats_cisp.loc['total_furtos', 'maximo'])} em um mês."),
]
for i, (nome, desc) in enumerate(leituras):
    y = 2.0 + i * 1.5
    cartao(s, 7.6, y, 5.1, 1.35)
    texto(s, 7.85, y + 0.15, 4.6, 0.35, nome, 16, COR_ROUBO, negrito=True)
    texto(s, 7.85, y + 0.52, 4.6, 0.8, desc, 13, COR_TEXTO_S)
notas(s, "Esta é a análise descritiva: resume os dados sem tentar explicar nada ainda. Média e mediana "
         "próximas indicam distribuição quase simétrica. O coeficiente de variação é o desvio padrão "
         "dividido pela média e permite comparar roubos e furtos. Quando olhamos delegacia por delegacia, "
         "a assimetria aumenta bastante, porque poucas delegacias têm valores muito altos.")

# --- 6. Análise descritiva: distribuição --------------------------
s = novo_slide()
cabecalho(s, "Análise descritiva · Distribuição",
          f"Furtos ficam perto de {mediana_furtos_mil} mil por mês; roubos variam mais")
rotulos_faixa = [dec(v / 1000, 1) for v in distribuicao.index]
for x, col, nome, cor in [(0.6, "total_roubos", "Roubos", COR_ROUBO), (6.8, "total_furtos", "Furtos", COR_FURTO)]:
    texto(s, x, 1.85, 5.9, 0.35, f"{nome}: quantidade de meses por faixa", 16, COR_TEXTO_S, negrito=True)
    g = colunas(s, x, 2.2, 5.9, 3.7, rotulos_faixa, distribuicao[col].tolist(), nome, cor)
    g.plots[0].gap_width = 15
    g.plots[0].data_labels.font.size = Pt(9)
    g.value_axis.visible = False
    g.value_axis.has_major_gridlines = False
    g.font.size = Pt(10)
texto(s, 0.6, 6.0, 12.1, 0.8,
      f"Metade dos meses ficou entre {num(est_r['q1'])} e {num(est_r['q3'])} roubos "
      f"(IQR de {num(est_r['iqr'])}) e entre {num(est_f['q1'])} e {num(est_f['q3'])} furtos "
      f"(IQR de {num(est_f['iqr'])}).", 16, COR_TEXTO_S)
texto(s, 0.6, 6.8, 12.1, 0.4, f"Cada coluna é uma faixa de {LARGURA_FAIXA} ocorrências; o rótulo é o início da "
      "faixa, em milhares. Mesma escala nos dois gráficos.", 11, COR_MUTED)
notas(s, "O histograma mostra quantos meses caíram em cada faixa. O intervalo interquartil (IQR) dos roubos é "
         "maior, porque eles passaram por períodos muito diferentes: alta, queda e retomada. Nos furtos a "
         "maior parte dos meses fica concentrada, com uma cauda baixa no início da série e uma cauda alta "
         "nos meses de Carnaval.")

# --- 7. Análise descritiva: composição ----------------------------
s = novo_slide()
cabecalho(s, "Análise descritiva · Composição",
          f"{dec(parte_rua, 0)}% dos roubos são na rua; {dec(parte_outros_furtos, 0)}% dos furtos caem em “outros”")
LIMITE_PARTICIPACAO = 1.0     # tipos abaixo de 1% ficam fora do gráfico
for x, tabela, nome, cor in [(0.6, composicao_roubos, "Roubos", COR_ROUBO),
                             (6.8, composicao_furtos, "Furtos", COR_FURTO)]:
    visiveis = tabela[tabela["participacao_%"] >= LIMITE_PARTICIPACAO]
    texto(s, x, 1.85, 5.9, 0.35, f"{nome}: participação de cada tipo", 16, COR_TEXTO_S, negrito=True)
    barras(s, x, 2.2, 5.9, 4.5, visiveis["tipo"].tolist(), visiveis["participacao_%"].tolist(), nome, cor)
texto(s, 0.6, 6.8, 12.1, 0.4, f"Participação no total de {ano_ini}–{ano_fim}. Tipos com menos de 1% não aparecem. "
      "“Na rua” soma transeunte, celular e coletivo.", 11, COR_MUTED)
notas(s, "A base traz subtipos de cada crime. Um cuidado: roubo de rua já é a soma de transeunte, celular e "
         "coletivo, então usamos só ele para não contar duas vezes; o código confere se os tipos somam "
         "exatamente o total. Quase metade dos furtos está em 'outros', o que limita a análise por tipo.")

# --- 8. Análise descritiva: concentração --------------------------
s = novo_slide()
cabecalho(s, "Análise descritiva · Onde e como se relacionam",
          f"Só 5 das {len(por_cisp)} delegacias concentram {dec(top5_furtos * 100, 0)}% dos furtos")
maiores = por_cisp.head(10)
rotulos_dp = [f"{c}ª DP" + (f" ({NOMES_DP[c]})" if c in NOMES_DP else "") for c in maiores["cisp"]]
texto(s, 0.6, 1.85, 7.4, 0.35, "10 delegacias com mais furtos no período", 16, COR_TEXTO_S, negrito=True)
barras(s, 0.6, 2.2, 7.4, 4.6, rotulos_dp, maiores["furtos"].tolist(), "Furtos", COR_FURTO, '#,##0')
destaque(s, 8.6, 2.0, 4.1, f"{dec(top5_furtos * 100, 0)}%", "dos furtos nas 5 maiores delegacias")
destaque(s, 8.6, 3.6, 4.1, f"{dec(top5_roubos * 100, 0)}%", "dos roubos nas 5 maiores delegacias", COR_ROUBO)
destaque(s, 8.6, 5.2, 4.1, f"r = {dec(corr_rf)}", "correlação entre roubos e furtos mensais", COR_MUTED)
notas(s, f"Roubos e furtos têm concentração parecida: poucas delegacias respondem por boa parte dos casos. A correlação de Pearson entre os totais mensais é "
         f"{dec(corr_rf)} (Spearman {dec(corr_spearman)}): fraca e negativa. Ou seja, um não acompanha o outro, "
         "e faz sentido estudar cada crime com o seu próprio modelo. Correlação não significa causa.")

# --- 9. Pergunta 1: roubos ---------------------------------------
s = novo_slide()
cabecalho(s, "Pergunta 1 · Os roubos caíram?",
          f"Os roubos caíram {pct(queda_roubos).lstrip('−')} até {ano_vale_roubo}, mas voltaram a subir")
destaque(s, 0.6, 2.0, 3.6, pct(queda_roubos), f"de {ano_pico_roubo} (pico) a {ano_vale_roubo} (menor valor)",
         COR_ROUBO)
destaque(s, 0.6, 3.7, 3.6, pct(retomada_roubos), f"de {ano_vale_roubo} a {ano_fim}", COR_ROUBO)
cartao(s, 0.6, 5.4, 3.6, 1.5)
texto(s, 0.85, 5.58, 3.1, 1.2,
      "A queda coincide com a implantação das UPPs (a 1ª em dez/2008). "
      "Coincidência no tempo não prova causa.", 13, COR_TEXTO_S)
colunas(s, 4.6, 1.8, 8.1, 5.1, anos_txt, anual["total_roubos"].tolist(), "Roubos", COR_ROUBO,
        destaque_idx=list(anual.index).index(ano_vale_roubo))
notas(s, f"Os roubos chegaram a {num(anual.loc[ano_pico_roubo, 'total_roubos'])} em {ano_pico_roubo} e "
         f"caíram para {num(anual.loc[ano_vale_roubo, 'total_roubos'])} em {ano_vale_roubo}. "
         "O período coincide com a política de pacificação (UPPs), mas os dados sozinhos não "
         "provam que ela foi a causa: outros fatores (economia, policiamento, registro) também "
         f"mudaram. A partir de {ano_vale_roubo + 1} os roubos voltam a crescer.")

# --- 10. Pergunta 2: sazonalidade ---------------------------------
s = novo_slide()
cabecalho(s, "Pergunta 2 · Existe época do ano?",
          f"{mes_pico_nome.capitalize()} é o mês com mais furtos: efeito Carnaval")
colunas(s, 0.6, 1.8, 7.9, 5.1, MESES, media_mes_furtos.round(0).tolist(), "Furtos", COR_FURTO,
        destaque_idx=mes_pico_furto - 1)
destaque(s, 9.0, 2.0, 3.7, pct(acima_media),
         f"furtos em {mes_pico_nome} em relação à média dos outros meses")
destaque(s, 9.0, 3.9, 3.7, f"{num(pico_cisp)} × {num(resto_cisp)}",
         f"mediana de furtos da {nome_cisp} em {mes_pico_nome} × demais meses", tamanho=40)
texto(s, 9.0, 6.4, 3.7, 0.5, f"Gráfico: média de furtos por mês do ano, {ano_ini}–{ano_fim}.", 11, COR_MUTED)
notas(s, f"A média de furtos em {mes_pico_nome} é {pct(acima_media)} maior do que nos outros "
         f"meses. O efeito é concentrado: a {nome_cisp} foi a delegacia com mais meses atípicos "
         "de furto, sempre perto do Carnaval, quando a região recebe grande público nos blocos.")

# --- 11. Pergunta 3: taxa por 100 mil -----------------------------
s = novo_slide()
cabecalho(s, "Pergunta 3 · Mais crime ou mais gente?",
          "Os furtos cresceram muito mais do que a população")
destaque(s, 0.6, 2.0, 3.6, pct(cresc_furtos), f"furtos registrados, {ano_ini}→{ano_fim}")
destaque(s, 0.6, 3.6, 3.6, pct(cresc_pop), f"população da cidade, {ano_ini}→{ano_fim}", COR_MUTED)
destaque(s, 0.6, 5.2, 3.6, pct(cresc_taxa_furtos), "taxa de furtos por 100 mil habitantes")
linhas_grafico(
    s, 4.6, 1.8, 8.1, 4.9, anos_txt,
    [("Roubos por 100 mil hab.", taxas[col_taxa_roubo].tolist(), COR_ROUBO),
     ("Furtos por 100 mil hab.", taxas[col_taxa_furto].tolist(), COR_FURTO)],
    marcadores=True,
)
texto(s, 4.6, 6.8, 8.1, 0.4, "Taxa = ocorrências no ano ÷ população (IBGE) × 100.000.", 11, COR_MUTED)
notas(s, "Para separar o efeito do crescimento da cidade, dividimos as ocorrências pela "
         f"população. Mesmo assim, a taxa de furtos variou {pct(cresc_taxa_furtos)}: o aumento não se "
         "explica só por haver mais gente.")

# --- 12. Outliers -----------------------------------------------
s = novo_slide()
cabecalho(s, "Outliers", "Encontrar os valores atípicos e decidir o que fazer")
metodos = [
    ("IQR (principal)", "Fora de Q1 − 1,5×IQR e Q3 + 1,5×IQR. Não exige distribuição normal."),
    ("Z-score > 3", "Mais de 3 desvios padrão da média. Usado só para comparação."),
    ("IQR por delegacia", "Compara cada mês com o histórico da própria delegacia."),
]
texto(s, 0.6, 1.85, 5.6, 0.4, "Como detectamos", 18, COR_TEXTO_S, negrito=True)
for i, (nome, desc) in enumerate(metodos):
    y = 2.4 + i * 1.45
    cartao(s, 0.6, y, 5.6, 1.25)
    texto(s, 0.85, y + 0.17, 5.1, 0.35, nome, 16, COR_ROUBO, negrito=True)
    texto(s, 0.85, y + 0.55, 5.1, 0.6, desc, 13, COR_TEXTO_S)
texto(s, 6.9, 1.85, 5.8, 0.4, "O que decidimos", 18, COR_TEXTO_S, negrito=True)
decisoes = [
    (f"{len(altos)} meses de furto muito altos",
     ("Todos em fevereiro ou março (" if altos_carnaval else "Meses: ") + ", ".join(altos_rotulos)
     + ("). São reais, efeito Carnaval: mantidos." if altos_carnaval else ". Mantidos.")),
    (f"{len(baixos)} meses de furto muito baixos",
     "Concentrados no início da série: refletem a tendência de alta, não erro. Mantidos."),
    ("Crimes raros", "Roubo a banco e a caixa eletrônico: o IQR é zero, então qualquer caso vira "
                     "outlier. Não interpretamos esses percentuais."),
]
for i, (nome, desc) in enumerate(decisoes):
    y = 2.4 + i * 1.45
    # "✓" não existe na Calibri; Segoe UI Symbol tem o símbolo
    circulo(s, 6.9, y + 0.1, 0.5, "✓" if i < 2 else "!", COR_FURTO, 16,
            "Segoe UI Symbol" if i < 2 else FONTE_CORPO)
    texto(s, 7.65, y + 0.05, 5.05, 0.35, nome, 16, COR_TEXTO_S, negrito=True)
    texto(s, 7.65, y + 0.43, 5.05, 0.9, desc, 13, COR_MUTED)
notas(s, "Detectar outliers é só metade do trabalho; a outra metade é decidir. Os picos de "
         "furto são reais (Carnaval) e por isso não foram removidos: eles são um achado. Os "
         "valores baixos do início da série só mostram que os furtos cresceram depois.")

# --- 13. Regressão: método ----------------------------------------
s = novo_slide()
cabecalho(s, "Pergunta 4 · Regressão linear", "O que a regressão acrescenta à análise descritiva")
texto(s, 0.6, 1.85, 12.1, 0.8, "y = b₀ + b₁·x₁ + b₂·x₂ + … + erro", 30, COR_ROUBO, negrito=True,
      fonte=FONTE_TITULO, alinhar=PP_ALIGN.CENTER)
texto(s, 0.6, 2.6, 12.1, 0.4, "Estima quanto y muda quando x muda, escolhendo os coeficientes que "
      "deixam o menor erro ao quadrado (MQO).", 15, COR_MUTED, alinhar=PP_ALIGN.CENTER)
modelos_txt = [
    ("Modelo 1 · Reta", "y = crimes no mês; x = tempo. Existe uma tendência única no período?"),
    ("Modelo 2 · Três fases", "A reta pode mudar de inclinação depois do pico e do vale dos roubos."),
    ("Modelo 3 · Múltipla", "Furtos explicados por tendência curva e por uma variável de Carnaval."),
]
for i, (nome, desc) in enumerate(modelos_txt):
    x = 0.6 + i * 4.1
    cartao(s, x, 3.2, 3.8, 1.75)
    texto(s, x + 0.3, 3.38, 3.2, 0.4, nome, 18, COR_TEXTO_S, negrito=True, fonte=FONTE_TITULO)
    texto(s, x + 0.3, 3.85, 3.2, 1.0, desc, 14, COR_MUTED)
leitura = [
    ("Coeficiente", "quanto y muda por unidade de x"),
    ("R²", "parcela da variação explicada"),
    ("p-valor", "efeito distinguível do acaso se < 0,05"),
    ("IC 95%", "faixa plausível do coeficiente"),
]
for i, (nome, desc) in enumerate(leitura):
    x = 0.6 + i * 3.08
    texto(s, x, 5.25, 2.9, 0.35, nome, 16, COR_FURTO, negrito=True)
    texto(s, x, 5.62, 2.85, 0.7, desc, 14, COR_TEXTO_S)
texto(s, 0.6, 6.8, 12.1, 0.4, f"Base: {len(reg)} totais mensais do município. Erros padrão HAC (Newey-West), "
      "que corrigem a autocorrelação da série.", 11, COR_MUTED)
notas(s, "Até aqui descrevemos o que aconteceu. A regressão quantifica: quanto os crimes mudam por mês e quanto "
         "pesa o Carnaval. Como meses vizinhos se parecem, os erros não são independentes; por isso usamos "
         "erros padrão HAC, de Newey-West. Eles não mudam os coeficientes, só tornam os p-valores e os "
         "intervalos de confiança mais honestos.")

# --- 14. Modelo 1: reta -------------------------------------------
s = novo_slide()
cabecalho(s, "Modelo 1 · Tendência simples",
          f"Uma reta explica {dec(R2_FURTOS_RETA * 100, 0)}% dos furtos, "
          f"mas só {dec(R2_ROUBOS_RETA * 100, 0)}% dos roubos")
anos_mes = [str(a) for a in reg["ano"]]
paineis_slide = [
    (0.6, "total_furtos", "ajuste_furtos_reta", "Furtos", COR_FURTO,
     f"{sinal(inc_furtos_reta)} furtos por mês · p {p_txt(m_furtos_reta.pvalues['t'])} · R² = {dec(R2_FURTOS_RETA)}"),
    (6.8, "total_roubos", "ajuste_roubos_reta", "Roubos", COR_ROUBO,
     f"{sinal(inc_roubos_reta)} roubos por mês · p = {p_txt(p_roubos_reta)} · R² = {dec(R2_ROUBOS_RETA)}"),
]
for x, y_col, ajuste, nome, cor, legenda in paineis_slide:
    g = linhas_grafico(s, x, 1.95, 5.9, 3.95, anos_mes,
                       [(nome, reg[y_col].tolist(), cor),
                        ("Reta ajustada", reg[ajuste].round(1).tolist(), COR_ESCURO)])
    pular_rotulos(g, 24)
    texto(s, x, 5.95, 5.9, 0.4, legenda, 15, COR_TEXTO_S, negrito=True)
texto(s, 0.6, 6.45, 12.1, 0.6, "A reta dos roubos não é estatisticamente distinguível de zero: a média do período "
      "esconde uma alta, uma queda e uma retomada.", 15, COR_MUTED)
notas(s, f"Na regressão simples, o coeficiente é a variação média por mês. Os furtos sobem cerca de "
         f"{num(inc_furtos_reta)} por mês. Nos roubos, a reta quase não explica nada e o p-valor é "
         f"{p_txt(p_roubos_reta)}, acima de 0,05. Isso não quer dizer que os roubos ficaram estáveis: "
         "o modelo é que está errado para esse padrão. É o gancho para o próximo slide.")

# --- 15. Modelo 2: três fases -------------------------------------
s = novo_slide()
cabecalho(s, "Modelo 2 · Roubos em três fases",
          f"Com três fases, o modelo passa a explicar {dec(R2_ROUBOS_FASES * 100, 0)}% dos roubos")
g = linhas_grafico(s, 0.6, 1.8, 8.3, 4.9, anos_mes,
                   [("Roubos", reg["total_roubos"].tolist(), COR_ROUBO),
                    ("Modelo de três fases", reg["ajuste_roubos_fases"].round(1).tolist(), COR_ESCURO)])
pular_rotulos(g, 12)
for i, linha in fases_roubos.iterrows():
    y = 1.95 + i * 1.6
    destaque(s, 9.3, y, 3.4, sinal(linha["roubos_por_mes"]),
             f"roubos por mês em {linha['fase']} (IC 95%: {sinal(linha['ic95_inf'])} a {sinal(linha['ic95_sup'])})",
             COR_ROUBO, tamanho=44)
texto(s, 0.6, 6.8, 12.1, 0.4, f"Quebras em dez/{ano_pico_reg} e dez/{ano_vale_reg}, escolhidas a partir do pico e do "
      "vale anuais; por isso os p-valores deste modelo são otimistas.", 11, COR_MUTED)
notas(s, "A regressão segmentada deixa a reta mudar de inclinação em dois pontos. Agora dá para medir as três "
         "fases: alta, queda e retomada. O modelo descreve quando o padrão mudou, mas não por quê: a queda "
         "coincide com as UPPs, e a regressão sozinha não prova essa relação. As datas de quebra foram "
         "escolhidas vendo os dados, então os testes de significância precisam ser lidos com cautela.")

# --- 16. Modelo 3: Carnaval ---------------------------------------
s = novo_slide()
cabecalho(s, "Modelo 3 · Regressão múltipla",
          f"O Carnaval soma cerca de {num(efeito_carnaval)} furtos ao mês em que acontece")
g = linhas_grafico(s, 0.6, 1.8, 7.4, 4.9, anos_mes,
                   [("Furtos", reg["total_furtos"].tolist(), COR_FURTO),
                    ("Modelo: curva + Carnaval", reg["ajuste_furtos_mult"].round(1).tolist(), COR_ESCURO)])
pular_rotulos(g, 12)
termos = [("Constante", "Intercept"), ("t (mês)", "t"), ("t²", "I(t ** 2)"), ("Carnaval", "carnaval")]
linhas_tabela = [["Termo", "Coef.", "p-valor"]]
linhas_tabela += [[nome, dec(m_furtos_mult.params[t], 2 if t == "I(t ** 2)" else 1),
                   p_txt(m_furtos_mult.pvalues[t])] for nome, t in termos]
tabela_slide(s, 8.4, 1.95, 4.3, linhas_tabela, [0.44, 0.3, 0.26], altura=0.45, tamanho=14)
destaque(s, 8.4, 4.35, 4.3, f"R² = {dec(R2_FURTOS_MULT)}", f"contra {dec(R2_FURTOS_RETA)} da reta simples", tamanho=40)
texto(s, 8.4, 5.75, 4.3, 1.0, f"IC 95% do Carnaval: {num(ic_carnaval[0])} a {num(ic_carnaval[1])} furtos. "
      f"Tendência atinge o topo perto de {topo_furtos_txt}.", 14, COR_TEXTO_S)
texto(s, 0.6, 6.8, 12.1, 0.4, "Carnaval = 1 no mês da terça de Carnaval (março em 2011 e 2014, fevereiro nos demais "
      "anos) e 0 nos outros meses.", 11, COR_MUTED)
notas(s, "Aqui colocamos três variáveis juntas. O t e o t² formam uma curva: os furtos sobem e depois param de "
         "subir. A variável Carnaval é uma dummy, que vale 1 só no mês da festa. Mantida a tendência, esse mês "
         f"tem cerca de {num(efeito_carnaval)} furtos a mais. Usar o mês real do Carnaval importa: em 2011 e 2014 "
         "ele foi em março, e foi justamente em março que apareceram os picos desses anos. O mesmo dado que "
         "tinha aparecido como outlier agora é explicado pelo modelo.")

# --- 17. Diagnóstico ----------------------------------------------
s = novo_slide()
dw_mult = resumo_modelos.loc["3. Furtos: curva+Carnaval", "durbin_watson"]
dw_fases = resumo_modelos.loc["2. Roubos: três fases", "durbin_watson"]
cabecalho(s, "Diagnóstico", "Os modelos são confiáveis? O que os resíduos mostram")
linhas_tabela = [["Modelo", "R²", "R² ajust.", "Erro médio", "Durbin-Watson"]]
linhas_tabela += [[nome, dec(l["r2"]), dec(l["r2_ajustado"]), num(l["erro_medio"]), dec(l["durbin_watson"])]
                  for nome, l in resumo_modelos.iterrows()]
tabela_slide(s, 0.6, 1.95, 7.3, linhas_tabela, [0.36, 0.11, 0.15, 0.16, 0.22], altura=0.5, tamanho=13)
g = colunas(s, 0.6, 4.55, 7.3, 2.2, anos_mes, reg["residuo_furtos_mult"].round(0).tolist(), "Resíduos", COR_FURTO)
g.plots[0].has_data_labels = False
g.plots[0].gap_width = 20
g.font.size = Pt(10)
g.category_axis.tick_label_position = XL_TICK_LABEL_POSITION.LOW
pular_rotulos(g, 12)
texto(s, 0.6, 6.8, 7.3, 0.4, "Gráfico: resíduos do modelo 3 (observado − modelo), mês a mês.", 11, COR_MUTED)
achados = [
    ("Erro médio", "Em quantas ocorrências por mês o modelo erra, em média. Caiu de um modelo para o outro."),
    (f"Modelo 3: DW = {dec(dw_mult)}", "Perto de 2: quase não resta padrão entre um mês e o seguinte."),
    (f"Modelo 2: DW = {dec(dw_fases)}", "Longe de 2: ainda há padrão não explicado. Os erros HAC compensam, mas o "
                                        "modelo é uma simplificação."),
]
for i, (nome, desc) in enumerate(achados):
    y = 1.95 + i * 1.6
    cartao(s, 8.3, y, 4.4, 1.45)
    texto(s, 8.55, y + 0.15, 3.9, 0.35, nome, 16, COR_ROUBO, negrito=True)
    texto(s, 8.55, y + 0.52, 3.9, 0.9, desc, 13, COR_TEXTO_S)
notas(s, "Um bom modelo deixa resíduos sem padrão. O teste de Durbin-Watson vai de 0 a 4 e fica perto de 2 quando "
         "não há autocorrelação. O modelo 3 está bem; o modelo 2 ainda tem autocorrelação, então serve para "
         "descrever as fases, não para prever. No notebook também estão os testes de Breusch-Pagan, que "
         "indica variância não constante (os erros HAC também corrigem isso), e de Jarque-Bera.")

# --- 18. O código -------------------------------------------------
s = novo_slide()
cabecalho(s, "No código", "Três passos para a regressão em Python")
codigo(s, 0.6, 1.9, 7.6, 4.75, [
    "import statsmodels.formula.api as smf",
    "",
    "# 1. Criar as variáveis explicativas",
    'reg["t"] = np.arange(len(reg))',
    'reg["carnaval"] = (reg["mes"] ==',
    '    reg["ano"].map(MES_CARNAVAL)).astype(int)',
    "",
    "# 2. Ajustar o modelo com erros HAC",
    'modelo = smf.ols(',
    '    "total_furtos ~ t + I(t ** 2) + carnaval",',
    '    data=reg,',
    ').fit(cov_type="HAC", cov_kwds={"maxlags": 12})',
    "",
    "# 3. Ler o resultado",
    "print(modelo.summary())",
], tamanho=14)
passos = [
    ("Preparar", "A base já vem do SQLite. Só criamos o contador de meses e a variável de Carnaval."),
    ("Ajustar", "A fórmula é escrita como no quadro: y ~ x₁ + x₂. O statsmodels faz o resto."),
    ("Interpretar", "O summary() traz coeficientes, p-valores, IC e R². Tudo vai para o Excel, o banco e estes slides."),
]
for i, (nome, desc) in enumerate(passos):
    y = 1.95 + i * 1.6
    circulo(s, 8.6, y, 0.6, str(i + 1), COR_FURTO, 16)
    texto(s, 9.4, y - 0.02, 3.3, 0.35, nome, 18, COR_TEXTO_S, negrito=True)
    texto(s, 9.4, y + 0.36, 3.3, 1.1, desc, 14, COR_MUTED)
notas(s, "Este é o coração da regressão no notebook (itens 11.2 a 11.6). O restante do código compara os modelos, "
         "gera os gráficos e guarda os resultados. Os números destes slides são calculados pelo próprio código: "
         "se a base mudar, a apresentação acompanha.")

# --- 19. Limitações --------------------------------------------
s = novo_slide()
cabecalho(s, "Limitações", "O que estes dados não conseguem dizer")
limites = [
    ("Registro não é crime",
     "A base conta boletins de ocorrência. Parte da alta dos furtos pode ser mais gente "
     "registrando (seguro, bloqueio de celular, delegacias mais acessíveis), e não mais crime."),
    ("Delegacias não são comparáveis",
     "Os números por delegacia são absolutos. O Centro recebe muita gente que não mora lá, "
     "então parece sempre “pior”. Por isso a taxa foi calculada só para a cidade."),
    ("Bicicleta só desde 2014",
     "Roubo e furto de bicicleta começaram a ser registrados em 2014. Antes disso o dado é "
     "vazio e ficou fora dos cálculos."),
    ("Regressão não prova causa",
     "Os modelos descrevem o passado. Não mostram por que os crimes mudaram e não foram "
     "feitos para prever os próximos anos."),
]
for i, (nome, desc) in enumerate(limites):
    x = 0.6 + i * 3.08
    cartao(s, x, 1.9, 2.85, 4.95)
    circulo(s, x + 0.3, 2.15, 0.6, "!", COR_ESCURO, 20)
    texto(s, x + 0.3, 2.9, 2.3, 0.9, nome, 18, COR_TEXTO_S, negrito=True, fonte=FONTE_TITULO)
    texto(s, x + 0.3, 3.75, 2.3, 3.0, desc, 15, COR_MUTED)
notas(s, "A limitação mais importante: os dados medem crimes registrados, não crimes "
         "cometidos. Isso não invalida a análise, mas muda como lemos as conclusões.")

# --- 20. Conclusões -----------------------------------------------
s = novo_slide(COR_ESCURO)
texto(s, 0.9, 0.7, 11, 0.4, "CONCLUSÕES", 14, COR_FURTO, negrito=True)
texto(s, 0.9, 1.1, 11.5, 0.9, "O que aprendemos", 38, COR_BRANCO, negrito=True, fonte=FONTE_TITULO)
conclusoes = [
    (pct(queda_roubos), f"Roubos caíram de {ano_pico_roubo} a {ano_vale_roubo} e depois voltaram a subir."),
    (pct(acima_media), f"Furtos em {mes_pico_nome}: o Carnaval é um pico que se repete todo ano."),
    (pct(cresc_taxa_furtos), "Taxa de furtos por habitante: o aumento vai além do crescimento da cidade."),
    (sinal(efeito_carnaval), f"Furtos no mês do Carnaval, segundo a regressão (R² = {dec(R2_FURTOS_MULT)})."),
    (f"R² {dec(R2_ROUBOS_FASES)}", "Três fases explicam os roubos: uma reta única esconde o padrão."),
    ("!", "As conclusões valem para crimes registrados, não para todos os crimes."),
]
for i, (numero, frase) in enumerate(conclusoes):
    y = 2.1 + i * 0.85
    texto(s, 0.9, y, 2.6, 0.7, numero, 30, COR_FURTO, negrito=True, fonte=FONTE_TITULO,
          ancora=MSO_ANCHOR.MIDDLE)
    texto(s, 3.6, y, 9.0, 0.7, frase, 19, COR_BRANCO, ancora=MSO_ANCHOR.MIDDLE)
notas(s, "Fechar retomando as quatro perguntas e a principal limitação.")

# --- 21. Fontes -----------------------------------------------------
s = novo_slide(COR_ESCURO)
texto(s, 0.9, 1.0, 11, 0.4, "OBRIGADO", 14, COR_FURTO, negrito=True)
texto(s, 0.9, 1.5, 11.5, 1.0, "Perguntas?", 50, COR_BRANCO, negrito=True, fonte=FONTE_TITULO)
texto(s, 0.9, 3.5, 11.5, 2.4, [
    "Dados: Instituto de Segurança Pública do RJ (ISP), BaseDPEvolucaoMensalCisp.csv",
    "População: IBGE/SIDRA, tabelas 6579, 793 e 202",
    "Ferramentas: Python, pandas, SQLite, statsmodels, matplotlib, python-pptx e openpyxl",
    "Código completo: notebook do projeto no Google Colab",
], 18, COR_SUAVE, espaco=10)
notas(s, "Abrir para perguntas. Se quiserem ver detalhes, voltar ao notebook e mostrar os itens 9.2 a 9.4 "
         "(descritiva) e 11.2 a 11.6 (regressão).")

prs.save(ARQUIVO_PPTX)
print("\nApresentação salva:", ARQUIVO_PPTX, f"({len(prs.slides)} slides)")

files.download(ARQUIVO_PPTX)
